The purpose of the dynamic pricing pipeline is to understand the way the current pricing model works by analyzing the database and create the best pipeline that mimics the exact nature of the pricing. 

In order to understand the database, we have come up with a series of tests, each of them with a specific purpose of understanding a specific fact about the database. 

After we run the tests, we analyse the outputs and then decide on the specific method to model the pricing

# The settings

In [ ]:
SEED = 42
EPS = 0.02
SEEDS = (0, 1, 2, 3, 4)
TEST_SIZE = 0.25

We split the dataset into 25% testing and 75% training. The seed is just the random shuffle that decides what 25% goes into testing.

Because we take the 25% randomly, when you create a linear regression model, or any type of model for that matter, the accuracy of the model on the testing dataset may differ a bit based on what chunk you train your model on. For example if you take the seed 42 you might get 0.878 accuracy on the test dataset, but if you take seed 43 you get 0.871. 

The `EPS` is just the wobble zone that accounts for this fact. If you train a different model and its absolute difference from the other model is < `EPS`, even if it's intrinsically better, it doesn't mean it is actually better, it's just the natural accuracy bounce based on the shuffle of the train and test set.

The `SEEDS` vector will be used on the tests that require a mean of different runs.

# The Columns

In [ ]:
TARGET = "Historical_Cost_of_Ride"
DURATION = "Expected_Ride_Duration"
VEHICLE = "Vehicle_Type"

DURATION_ONLY = [DURATION]
BASE_FEATURES = [DURATION, VEHICLE]

EXTRA_FEATURES = [
    "Number_of_Riders",
    "Number_of_Drivers",
    "Location_Category",
    "Customer_Loyalty_Status",
    "Number_of_Past_Rides",
    "Average_Ratings",
    "Time_of_Booking",
]
NUMERIC_EXTRAS = [
    "Number_of_Riders",
    "Number_of_Drivers",
    "Number_of_Past_Rides",
    "Average_Ratings",
]
ALL_FEATURES = EXTRA_FEATURES + BASE_FEATURES

CATEGORICAL = ["Location_Category", "Customer_Loyalty_Status", "Time_of_Booking", VEHICLE]

The `TARGET` is the column that we want to predict. 

Because we will see later that the entire model is based solely on the duration of the ride and the type of vehicle, we have separated those columns into the `BASE_FEATURES`. The other columns are the extra information that will play an important role in the tests and the decision of our model.

We have also created the `CATEGORICAL`, which are the columns that are not represented by numbers, and we will need to transform them into numbers with one hot encoding

# Data/pipeline

In [ ]:
def load_data() -> pd.DataFrame:
    return pd.read_csv(DATA_PATH)


def make_pipeline(cols, model):
   
    cat = [c for c in cols if c in CATEGORICAL]
    num = [c for c in cols if c not in CATEGORICAL]
    pre = ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
            ("num", "passthrough", num),
        ]
    )
    return Pipeline([("pre", pre), ("model", model)])


def split_data(df: pd.DataFrame, *, seed: int = SEED, test_size: float = TEST_SIZE):
    
    return train_test_split(df, test_size=test_size, random_state=seed)

The `load_data()` function is just taking the dataset csv and transforming it into a pandas database so that it can be easily parsed and modeled

The `make_pipeline()` function is just the skeleton of creating a pipeline, without actually specifying the actual cols and the models that we will use. We run through every column of the cols that we want to use, and if it is numerical based we put it into the numerical cols, and if they are categorical, we store them separately because they will need further modifications in order to be used. We do these separations by the columns of the database described in a step above.

Now using the `pre`, we prepare the cols in order to actually be used by the models. The numerical ones are just parsed through unchanged. The categorical ones need one hot encoding

The models are intrinsically just mathematical models defined by analysing the discrepancies between numbers. But categorical cols are not defined by numbers, so they cannot be directly used by models, so we need to find a way to transform them into numbers. We cannot simply just choose a different number for each type of a category, because those numbers will be interpreted as weights. For example for the `Location_Category` if you have 1 for `is_Urban`, 2 for `is_Suburban` and so on, the model will consider that `is_Suburban` is 2 times more important than `is_Urban`. The idea behind one hot encoding is that you create a yes/no column for each category. In this way `is_Urban` and `is_Suburban` are just as important to the model

The return is just the modified assembly, such that you can just run `predict` or `fit` on the already pre-configured data.

The `split_data` just creates the test and the train datasets based on the above mentioned seed and size, in this case 25% test and 75% train, on seed 42. These are industry standards.

In [ ]:

def held_out_r2(train_df, test_df, cols, model) -> float:
    pipe = make_pipeline(cols, clone(model)).fit(train_df[cols], train_df[TARGET])
    return float(r2_score(test_df[TARGET], pipe.predict(test_df[cols])))


def held_out_r2_seed(df, cols, model, *, seed: int = SEED) -> float:
    train_df, test_df = split_data(df, seed=seed)
    return held_out_r2(train_df, test_df, cols, model)


def mean_held_out_r2(df, cols, model, seeds=SEEDS) -> float:
    return float(np.mean([held_out_r2_seed(df, cols, clone(model), seed=s) for s in seeds]))

The `held_out_r2()` creates a pipeline with the model specified a step before with an untrained clone of the model and asks it to fit the train dataset training cols to predict the target col.

It returns the `r2_score` of the trained model on the test cols.

the r2 score is basically 1 - (residual sum of squares) / (total sum of squares)

more specifically, for this case

for every ride, you calculate the real price - the predicted price, square this difference and sum it to the total sum

the total sum of squares is basically calculating how good you can approximate the price by just taking the mean of the prices. It is basically for every ride, you take the real price - the mean of the prices, you square this difference and add it to the total sum

now, when you divide this total sum, you basically calculate how good the model predicted the prices in relation to just taking the mean of the prices. If r^2 = 0.93 for example, this means the model eliminates approximately 93% of the prediction error made by simply predicting the mean.

The `held_out_r2_seed()` is the same as `held_out_r2`, but you specify the seed by which you divide the dataset.

the `mean_held_out_r2()` is just running the model on multiple seeds and taking the mean of the r^2 to take into account the wobble generated by the dataset split.

In [ ]:
def per_vehicle_fit(df: pd.DataFrame) -> dict[str, tuple[float, float]]:
    params: dict[str, tuple[float, float]] = {}
    for vehicle, group in df.groupby(VEHICLE):
        lr = LinearRegression().fit(group[[DURATION]], group[TARGET])
        params[vehicle] = (float(lr.coef_[0]), float(lr.intercept_))
    return params


def formula_predict(df: pd.DataFrame, params: dict[str, tuple[float, float]]) -> pd.Series:
    slope = df[VEHICLE].map(lambda v: params[v][0])
    intercept = df[VEHICLE].map(lambda v: params[v][1])
    return intercept + slope * df[DURATION]


def cost_per_minute(df: pd.DataFrame) -> pd.Series:
    return df[TARGET] / df[DURATION]


Based on the tests (that we will discuss after this part) we came to the conclusion that the best model that predicts the existing pricing strategy is just a linear regression on duration and vehicle type.

Linear regression is just the line that predicts best the dots represented by the input.

the formula of a line is y = ax + b where a is the slope and b is the intercept

for our model, you have actually 2 lines, one for each vehicle type. The slope is ~ the same for both, the only different thing is the intercept.

the slope represents the per-minute rate and the intercept represents where the pricing starts from

now, for the code

in `per_vehicle_fit()` we firstly group the rides by vehicle types, because this is what affects the intercept, the premium vehicle will have a higher starting point

now for each vehicle group, we do a linear regression that fits duration on the price, and then read the slope and intercept.

now, with `formula_predict()`, you can now calculate ride prices without actually using a model. you just apply the line formula for your vehicle type and duration

the `cost_per_minute()` just calculates the cost/minute. One note here, because you have an intercept, the price is a bit inflated. if price = base + rate*duration, then price/duration = base/duration + rate, meaning that for short rides it looks pricier per minute.

This is how the price is actually calculated based on the dataset, but we have not explained yet why we use this method exactly. We have come to the conclusion that other more complex models are either worse or the same + unjustified complexity, and this happened after a series of tests designed specifically to observe these things. 

In continuation we will explain the tests and the results that led us to deduce that this is actually the best price predictor.

# Test 1 - Duration alone explains most

In [ ]:
def test_duration_alone_explains_most(train_df, test_df):
    r2 = pricing.held_out_r2(train_df, test_df, pricing.DURATION_ONLY, LinearRegression())
    assert r2 > 0.80, f"duration-only held-out R^2 = {r2:.4f}, expected > 0.80"


def test_linear_is_enough(data):
   
    linear = pricing.mean_held_out_r2(data, pricing.BASE_FEATURES, LinearRegression())
    gbm = pricing.mean_held_out_r2(
        data, pricing.BASE_FEATURES, GradientBoostingRegressor(random_state=pricing.SEED)
    )
    assert gbm - linear < pricing.EPS, (
        f"GBM ({gbm:.4f}) beat linear ({linear:.4f}) by more than EPS={pricing.EPS} "
        "-- there would be real curvature to model"
    )

in `test_duration_alone_explains_most()` we use the explained-above `held_out_r2()` using in the pipeline only the duration col with a linear regression model. if r^2 is > 0.8, it means that you can remove 80% of the prediction error made by simply predicting the mean solely by using a linear regression on the duration col.

now, we have also compared the effectiveness of a Gradient Boosting Regressor compared to a linear regression

How the Gradient Boosting Regressor works is a sequence of small decision trees. The central idea is that the new prediction = the old prediction + small correction

First you start with the mean, because for squared-error regression, the best constant initial prediction is the mean

After that you calculate the residuals, which is the actual price of each ride in our example - the mean

Now you create a split based on the feature you choose, for example in our case duration. This produces two leaves, one leaf where the duration is > the threshold, and one leaf where the durations are < threshold. Now for each leaf you compute the average of the residuals and then for each leaf for each row you compute the old prediction + the average of the residuals in that leaf times a learning rate

then with the new predictions, you calculate the new residuals and repeat the process for n times

after n times, with the last predictions you calculate the r^2 and calculate the absolute difference between it and the linear regression r^2, if it's bigger than the eps, it shows that the scores differ meaningfully according to your chosen tolerance. It does not automatically prove statistical significance.

now about the actual split, on how it is calculated, it tests possible thresholds and chooses the one that most reduces squared residual error.

More specifically, the tree considers candidate thresholds between consecutive sorted values. these come from the midpoints, assuming the data is sorted. then it does the process of splitting and calculating for each leaf the total squared error, and adds the two. for each split it does the same, and sees which one is the lowest.

If several features are supplied, the tree considers splits across all available features, not only one chosen feature. it takes into consideration all splits across all features and chooses the best one at each step.

# Test Extras Add nothing

In [ ]:
def test_extras_dont_help(data):
    base = pricing.mean_held_out_r2(data, pricing.BASE_FEATURES, LinearRegression())
    full = pricing.mean_held_out_r2(data, pricing.ALL_FEATURES, LinearRegression())
    assert full - base < pricing.EPS, (
        f"extras raised R^2 from {base:.4f} to {full:.4f} (>{pricing.EPS}) "
        "-- they would carry real signal"
    )

This one is pretty straightforward, you run two separate linear regressions, one solely on the base features and one with all the columns. When you compute the absolute difference on the r^2s, the result is < `EPS`

In [ ]:
def test_line_beats_gbm_on_everything(data):
    line = pricing.mean_held_out_r2(data, pricing.BASE_FEATURES, LinearRegression())
    gbm_all = pricing.mean_held_out_r2(
        data, pricing.ALL_FEATURES, GradientBoostingRegressor(random_state=pricing.SEED)
    )
    assert line >= gbm_all - pricing.EPS, (
        f"line on [duration, vehicle] ({line:.4f}) lost to GBM-on-everything "
        f"({gbm_all:.4f}) by more than EPS={pricing.EPS}"
    )

Here the concept is similar to the `linear_is_enough` test, but we run the GBM on all the features instead of just the base ones. This was made to see if there are any hidden patterns that a linear model could not catch.

In [ ]:
def test_extras_no_better_than_random(data, train_df, test_df):
    rng = np.random.default_rng(pricing.SEED)
    random_col = "__random__"
    cols = pricing.ALL_FEATURES + [random_col]

    train = train_df.assign(**{random_col: rng.normal(size=len(train_df))})
    test = test_df.assign(**{random_col: rng.normal(size=len(test_df))})

    model = GradientBoostingRegressor(random_state=pricing.SEED)
    pipe = pricing.make_pipeline(cols, model).fit(train[cols], train[pricing.TARGET])
    result = permutation_importance(
        pipe, test[cols], test[pricing.TARGET],
        n_repeats=10, random_state=pricing.SEED, scoring="r2",
    )
    importance = dict(zip(cols, result.importances_mean))

    random_importance = importance[random_col]
    assert importance[pricing.DURATION] > random_importance + 10 * pricing.EPS

    for feature in pricing.EXTRA_FEATURES:
        assert importance[feature] <= random_importance + pricing.EPS, (
            f"{feature} importance {importance[feature]:.4f} exceeded the random "
            f"column ({random_importance:.4f}) by more than EPS={pricing.EPS}"
        )

`rng = np.random.default_rng(SEED)` is a seeded random-number generator. We make copies of the train and test datasets in which we assign a new column called `__random__` filled with random numbers that follow a normal distribution. So the random column has approximately mean 0 and sd 1. We pick the GB model and train it on all the cols, including the decoy. `permutation_importance` measures how much the model depends on each col. For each col, we shuffle it and see how much the accuracy drops. A big drop means that the model relied a lot on it. We run it 10 times for each column and calculate the mean. `importance = dict(zip(cols, result.importances_mean))` maps each col to its importance. `assert importance[pricing.DURATION] > random_importance + 10 * pricing.EPS` calculates if the duration importance beats the random importance by 10x the eps.

then for each feature, we test if the importance of the feature is bigger than random importance + eps. if it fails, it means it's not more important than a col with random values

In [ ]:
def test_extras_not_significant(data):
    price = data[pricing.TARGET]
    for feature in pricing.NUMERIC_EXTRAS:
        r, p = pearsonr(data[feature], price)
        assert abs(r) < 0.1, f"{feature}: |r| = {abs(r):.4f}, expected < 0.1"
        assert p > 0.05, f"{feature}: p = {p:.4f}, expected > 0.05 (no real effect)"

This test might answer the same question as the previous one. but there is a nuance, and we kept it as one more might make things even clearer.

For each numerical feature, you calculate the pearson correlation coefficient, which tells you if there is a linear relationship between that col and the target col. -1 ≤ r ≤ 1, 1 means strong linear relationship, 0 means no linear relationship and -1 means strong negative linear relationship. The p-value tells us how likely it would be to observe a correlation at least this extreme, assuming there is no linear relationship. 

how you calculate r is by taking the mean of the feature you want to test and for each row you do the value - the mean. you do the same for the target col. and the formula is basically the sum of products of these rows / the square root of the sum of the (feature value - the mean) squared times the sum of the (target value - the mean) squared. The main idea is that matching signs produce positive correlation, opposite signs produce negative correlation and mixed patterns cancel toward zero.

# Test positive controls

In [ ]:
def test_vehicle_type_matters(data):
    cpm = pricing.cost_per_minute(data)
    economy = cpm[data[pricing.VEHICLE] == "Economy"]
    premium = cpm[data[pricing.VEHICLE] == "Premium"]

    t_stat, p_value = ttest_ind(economy, premium, equal_var=False)
    assert p_value < 1e-3, f"vehicle cost/min t-test p = {p_value:.2e}, expected < 1e-3"
    assert abs(economy.mean() - premium.mean()) > 0.5, "cost/min gap should be large"


This uses the same ideas as the previous test to determine if the type of vehicle matters. you split the data set into economy and premium, and for each you calculate the price/minute. now we want to know if that difference is just luck or noise, or if it is indeed a correlation. We run a t-test on the two groups. A t-test takes the averages of the two groups and gives us the p-value, that we have explained above, meaning if the two groups have the same price/min indeed, how often would luck produce a gap this big. Here the chance is near 0. And by comparing the mean we can see that the gap is not negligible, and thus we need two linear models for each type of car.

In [ ]:
def test_duration_matters(data):
    r, _ = pearsonr(data[pricing.DURATION], data[pricing.TARGET])
    assert r > 0.8, f"corr(duration, price) = {r:.4f}, expected > 0.8"

The same pearson correlation coefficient helps us to identify if the duration is indeed relevant.

# Test Reconstruct generator

In [ ]:
def test_recover_rate(data):
    params = pricing.per_vehicle_fit(data)
    economy_slope, economy_intercept = params["Economy"]
    premium_slope, premium_intercept = params["Premium"]

    assert abs(economy_intercept) < 10, f"Economy base fare {economy_intercept:.1f} not ~0"
    assert 30 < premium_intercept < 60, f"Premium base fare {premium_intercept:.1f} not ~46"
    assert abs(economy_slope - premium_slope) < 0.3, "per-minute slopes should match"
    assert 3.0 < economy_slope < 4.0 and 3.0 < premium_slope < 4.0


Here we just take the `per_vehicle_fit()` function we have explained at the beginning and test if the values are indeed what we assumed. We basically test our assumption that the intercepts are different but the slopes are basically the same

In [ ]:
def test_formula_reproduces_data(train_df, test_df):

    from sklearn.metrics import r2_score

    params = pricing.per_vehicle_fit(train_df)
    formula_pred = pricing.formula_predict(test_df, params)
    formula_r2 = r2_score(test_df[pricing.TARGET], formula_pred)

    gbm_r2 = pricing.held_out_r2(
        train_df, test_df, pricing.ALL_FEATURES,
        GradientBoostingRegressor(random_state=pricing.SEED),
    )
    assert abs(formula_r2 - gbm_r2) < pricing.EPS, (
        f"hand formula R^2 {formula_r2:.4f} differs from GBM-on-everything "
        f"{gbm_r2:.4f} by more than EPS={pricing.EPS}"
    )



We use this to test our arithmetic formula against an actual model. We load the params from pricing and construct the pricing on the test. Then we do the same prediction using all features with a GB model. we calculate the r^2s for both ways and do the absolute difference. If it's smaller than the `EPS` then it means that our arithmetic model is comparable to the GB in accuracy.

In [ ]:
def test_residuals_are_noise(data):
    params = pricing.per_vehicle_fit(data)
    residual = data[pricing.TARGET] - pricing.formula_predict(data, params)

    for feature in pricing.NUMERIC_EXTRAS:
        r, _ = pearsonr(residual, data[feature])
        assert abs(r) < 0.15, f"residual has structure vs {feature}: |r| = {abs(r):.4f}"
        assert r ** 2 < 0.02, (
            f"{feature} explains {r ** 2:.1%} of residual variance -- more than a whisper"
        )

For this test, we calculate the residuals by using our arithmetic model. The residuals are calculated by the actual target price - our predicted price.

then for each extra numeric feature, we calculate whether there's any linear correlation between the feature and the residuals. The idea is that if my formula captured everything real, then the residual should be noise and not related to any other feature. by taking the r^2 you can transform that into a %.

What we really want to test is: if we know for example the `Number_of_Drivers`, does that help us to guess if the residual will be big or small. Let's say that r is 0.14, that means r^2 is 0.02. that means the `Number_of_Drivers` explains 2% of the residual wiggle. Because our arithmetic model is not 100% perfect, it might overshoot or undershoot a bit based on the ride. The wiggle refers to this exact variation of the error.

The test then requires this to stay under 2% for every extra feature. Because it does, the residuals really are noise. No leftover feature can explain what the formula got wrong, which confirms the '+ noise' part of the thesis.

# Stats

## Pricing: the recovered generator

`price = base[vehicle] + slope[vehicle] * Expected_Ride_Duration + noise`

| Vehicle | Base fare | Per-minute rate |
|---------|-----------|------------------|
| Economy | ~$0 (fit: -$5.53) | $3.56/min |
| Premium | ~$46 (fit: $46.15) | $3.50/min |

The per-minute rate is essentially identical across vehicles; the only real
difference is Premium's flat base fare.

## Model comparison (held-out R²)

| Model | R² |
|-------|-----|
| Duration only (linear) | 0.850 |
| Hand formula (base + slope·duration) | 0.869 |
| Line on [duration, vehicle] (mean, 5 seeds) | 0.878 |
| Gradient boosting on **all** features (mean, 5 seeds) | 0.870 |

The main takeaway of our entire analysis is the idea that the whole pricing model of this database stands solely on the vehicle type and duration and can be created with arithmetic linear systems. In addition, we have demonstrated why this is the case and why adding any extra feature or a more powerful model does not add any more accuracy to the prediction.